# Task 2: measured results — CPU only

After GPU_WORK_COMPLETE, use Hardware accelerator **None**. This makes tables/plots from saved evidence and a qualitative worksheet. No quality judgments or experimental values are invented.

In [ ]:
from pathlib import Path
from uuid import uuid4
import hashlib
import json
import shutil
import signal
import subprocess
import sys
import tempfile
import time
import zipfile


def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(2**20), b''):
            h.update(block)
    return h.hexdigest()


def safe_extract(z, destination):
    destination = Path(destination).resolve()
    names = set()
    for item in z.infolist():
        target = (destination / item.filename).resolve()
        if destination not in target.parents or item.filename in names or (item.external_attr >> 16) & 0o170000 == 0o120000:
            raise RuntimeError('Unsafe or duplicate archive member')
        names.add(item.filename)
    z.extractall(destination)


def run_visible(command, cwd=None):
    process = subprocess.Popen(list(map(str, command)), cwd=cwd, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, bufsize=1)
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        code = process.wait()
    except KeyboardInterrupt:
        # The GPU orchestrator stops its child process and saves completed work.
        process.send_signal(signal.SIGINT)
        remaining, _ = process.communicate()
        print(remaining, end='', flush=True)
        code = process.returncode
    except BaseException:
        process.terminate()
        process.wait()
        raise
    if code:
        raise RuntimeError('Process failed; keep the runtime available and inspect the printed error.')


def latest_receipt(pattern, status, override=''):
    choices = [Path(override)] if override else sorted(Path('/content/drive/MyDrive/ATML-Assignment-2/task2').glob(pattern), reverse=True)
    for path in choices:
        record = json.loads(path.read_text())
        if record['status'] == status:
            return path, record
    raise RuntimeError('No completed preparation receipt found. Run the CPU preparation notebook first.')


def restore_project(ready, repo):
    spec = ready['project_archive']
    repo = Path(repo)
    with tempfile.TemporaryDirectory(prefix='task2_bootstrap_') as tmp:
        local = Path(tmp) / 'project.zip'
        shutil.copyfile(spec['path'], local)
        if file_sha(local) != spec['sha256']:
            raise RuntimeError('Saved project archive checksum differs')
        with zipfile.ZipFile(local) as z:
            manifest = json.loads(z.read('TASK2_BACKUP_MANIFEST.json'))['files_sha256']
            expected = set(manifest) | {'TASK2_BACKUP_MANIFEST.json'}
            if set(z.namelist()) != expected or len(z.namelist()) != len(expected):
                raise RuntimeError('Project archive inventory differs')
            for name, digest in manifest.items():
                if hashlib.sha256(z.read(name)).hexdigest() != digest:
                    raise RuntimeError('Project member checksum differs: ' + name)
            extracted = Path(tmp) / 'extracted'
            safe_extract(z, extracted)
        for name, digest in manifest.items():
            target = repo / name
            if target.exists() and file_sha(target) != digest:
                raise RuntimeError('Existing project file differs; use a fresh runtime: ' + name)
        for name in manifest:
            target = repo / name
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copyfile(extracted / name, target)

# Hardware accelerator: None. Optional exact ready receipt / reviewed CSV path:
READY_RECEIPT = ''
REVIEW_CSV = ''
import torch
from google.colab import drive
if torch.cuda.is_available():
    raise RuntimeError('Use Hardware accelerator: None for tables, plots and review.')
drive.mount('/content/drive')
ready_path, ready = latest_receipt('gpu_ready/*/ready.json', 'GPU_JOBS_READY', READY_RECEIPT)
experiment = Path(ready['experiment_root'])
progress = json.loads((experiment / 'progress.json').read_text())
if progress['status'] != 'GPU_WORK_COMPLETE':
    raise RuntimeError('GPU work is incomplete: ' + progress['status'] + '. Resume the GPU launcher first.')
repo = Path('/content/ATML-PA2-Task2')
restore_project(ready, repo)
run_visible([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '-r', repo / 'requirements.txt'])
from datetime import datetime, timezone
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '_' + uuid4().hex[:8]
local = Path('/content') / ('task2_evidence_' + stamp)
command = [sys.executable, '-u', '-m', 'task2_ppo.aggregate_cpu', '--experiment', experiment, '--output', local]
if REVIEW_CSV:
    command.extend(['--review', REVIEW_CSV])
run_visible(command, cwd=repo)
sys.path.insert(0, str(repo))
from task2_ppo.storage import commit_directory, publish_directory
committed = Path('/content') / ('task2_results_' + stamp)
commit_directory(local, committed)
receipt = publish_directory(committed, experiment / 'cpu_results')
print('RESULTS BACKUP VERIFIED:', experiment / 'cpu_results' / receipt['archive'], flush=True)
print('Extract the results ZIP to inspect plots/tables and complete qualitative_review.csv.', flush=True)
drive.flush_and_unmount()
print('CPU results saved and Drive flushed. No GPU was used.', flush=True)
